# End to End Pipeline

Runs the full lakehouse path in this notebook: Bronze verify, Silver rebuild, Type 2 versioning, Gold rebuild, and quality gates. Each stage asserts its contract before the next stage starts. Parquet outputs land in the same `data` paths the pipeline scripts use.

## Setup

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
BRONZE = ROOT / "data" / "bronze"
SILVER = ROOT / "data" / "silver"
GOLD = ROOT / "data" / "gold"

from src.gold.build import (
    build_fact_orders,
    build_fact_returns,
    mart_ramadan_seasonality,
    mart_return_rate_by_product,
    mart_revenue_by_store_month,
)
from src.ops.db import with_batch
from src.quality.gates import check_not_null, check_referential, check_unique, score
from src.silver.scd2 import CUSTOMER_TRACKED, EMPLOYEE_TRACKED, apply_scd2
from src.silver.transforms import (
    clean_customers,
    clean_orders,
    clean_products,
    drop_extract_meta,
    enrich_returns,
    union_orders,
)

BATCH = "notebook"
timings = {}
print("imports ok")

imports ok


## Stage 1 Bronze verify

All nine landing tables present with rows.

In [2]:
t0 = time.time()
expected = [
    "dim_customers",
    "dim_date",
    "dim_stores",
    "dim_employees",
    "dim_products",
    "fact_orders_2022_2023",
    "fact_orders_2024",
    "fact_order_details",
    "fact_returns",
]
bronze = {}
for name in expected:
    path = BRONZE / f"{name}.parquet"
    assert path.exists(), f"missing {path}"
    bronze[name] = pd.read_parquet(path)
    assert len(bronze[name]) > 0, f"empty {name}"
timings["bronze"] = time.time() - t0
print(
    f"bronze ok: {len(bronze)} tables, "
    + f"{sum(len(df) for df in bronze.values())} rows, {timings['bronze']:.1f}s"
)

bronze ok: 9 tables, 42877 rows, 0.1s


## Stage 2 Silver rebuild

Clean every table with the same functions the Silver script uses.

In [3]:
t0 = time.time()
silver = {}
silver["dim_customers"] = clean_customers(bronze["dim_customers"])
silver["dim_products"] = clean_products(bronze["dim_products"])
for name in ["dim_stores", "dim_employees", "dim_date", "fact_order_details"]:
    silver[name] = drop_extract_meta(bronze[name])
silver["fact_orders"] = union_orders(
    clean_orders(bronze["fact_orders_2022_2023"]),
    clean_orders(bronze["fact_orders_2024"]),
)
silver["fact_returns"] = enrich_returns(
    drop_extract_meta(bronze["fact_returns"]),
    drop_extract_meta(bronze["fact_order_details"]),
)
assert silver["dim_customers"]["customer_id"].isna().sum() == 0
assert silver["fact_orders"].duplicated(subset=["order_id"]).sum() == 0
for name, df in silver.items():
    with_batch(df, BATCH).to_parquet(SILVER / f"{name}.parquet", index=False)
timings["silver"] = time.time() - t0
print(f"silver ok: {len(silver)} tables, {timings['silver']:.1f}s")

silver ok: 8 tables, 0.2s


## Stage 3 Type 2 versioning

Anchor versions on the earliest order date, same rule as the SCD2 script.

In [4]:
t0 = time.time()
parsed = pd.to_datetime(silver["fact_orders"]["order_date"])
anchor = parsed.min().date().isoformat()
change = parsed.max().date().isoformat()
scd2 = {}
for table, key, tracked, sk in [
    ("dim_customers", "customer_id", CUSTOMER_TRACKED, "customer_sk"),
    ("dim_employees", "employee_id", EMPLOYEE_TRACKED, "employee_sk"),
]:
    path = SILVER / f"{table}_scd2.parquet"
    current = pd.read_parquet(path) if path.exists() else None
    start = anchor if current is None else change
    out = apply_scd2(current, silver[table], key, tracked, sk, start)
    assert out["is_current"].sum() >= silver[table][key].nunique()
    with_batch(out, BATCH).to_parquet(path, index=False)
    scd2[table] = out
timings["scd2"] = time.time() - t0
print(f"scd2 ok: anchor={anchor}, {timings['scd2']:.1f}s")

scd2 ok: anchor=2022-01-01, 0.7s


## Stage 4 Gold rebuild

Facts plus all three marts from the rebuilt Silver tables.

In [5]:
t0 = time.time()
gold = {}
for name in ["dim_date", "dim_stores", "dim_products"]:
    gold[name] = silver[name]
gold["dim_customers"] = scd2["dim_customers"]
gold["dim_employees"] = scd2["dim_employees"]
gold["fact_orders"] = build_fact_orders(
    silver["fact_orders"], gold["dim_customers"], gold["dim_employees"]
)
gold["fact_returns"] = build_fact_returns(silver["fact_returns"], gold["fact_orders"])
gold["fact_order_details"] = silver["fact_order_details"]
gold["mart_revenue_by_store_month"] = mart_revenue_by_store_month(gold["fact_orders"])
gold["mart_return_rate_by_product"] = mart_return_rate_by_product(
    gold["fact_order_details"], gold["fact_returns"]["order_id"]
)
gold["mart_ramadan_seasonality"] = mart_ramadan_seasonality(
    gold["fact_orders"], gold["dim_date"]
)
assert len(gold["fact_orders"]) == len(silver["fact_orders"])
assert all(len(gold[name]) > 0 for name in gold)
for name, df in gold.items():
    with_batch(df, BATCH).to_parquet(GOLD / f"{name}.parquet", index=False)
timings["gold"] = time.time() - t0
print(f"gold ok: {len(gold)} tables, {timings['gold']:.1f}s")

gold ok: 11 tables, 0.2s


## Stage 5 Quality gates

Null, uniqueness, and referential suite over the rebuilt Gold. Fails the run when any key contract breaks.

In [6]:
results = {}
results.update(
    {
        f"null_{k}": v
        for k, v in check_not_null(
            gold["fact_orders"], ["order_id", "customer_sk"]
        ).items()
    }
)
results.update(
    {
        f"unique_{k}": v
        for k, v in check_unique(gold["fact_orders"], ["order_id"]).items()
    }
)
results["ref_customer"] = check_referential(
    gold["fact_orders"], "customer_id", gold["dim_customers"], "customer_id"
)
results["ref_product"] = check_referential(
    gold["fact_order_details"], "product_id", gold["dim_products"], "product_id"
)
pct, failed = score(results)
print(f"gates score: {pct:.1f} percent, failed: {failed}")
gate_df = pd.DataFrame([{"check": k, "failures": v} for k, v in results.items()])
print(gate_df.to_string(index=False))
assert results["unique_order_id"] == 0
assert results["null_order_id"] == 0

gates score: 100.0 percent, failed: 0
           check  failures
   null_order_id         0
null_customer_sk         0
 unique_order_id         0
    ref_customer         0
     ref_product         0


## Run summary

Stage timings and row counts for this run.

In [7]:
summary = pd.DataFrame(
    [
        {
            "stage": "bronze",
            "tables": len(bronze),
            "rows": sum(len(df) for df in bronze.values()),
            "seconds": round(timings["bronze"], 1),
        },
        {
            "stage": "silver",
            "tables": len(silver),
            "rows": sum(len(df) for df in silver.values()),
            "seconds": round(timings["silver"], 1),
        },
        {
            "stage": "scd2",
            "tables": len(scd2),
            "rows": sum(len(df) for df in scd2.values()),
            "seconds": round(timings["scd2"], 1),
        },
        {
            "stage": "gold",
            "tables": len(gold),
            "rows": sum(len(df) for df in gold.values()),
            "seconds": round(timings["gold"], 1),
        },
    ]
)
summary

,stage,tables,rows,seconds
0,bronze,9,42877,0.1
1,silver,8,42827,0.2
2,scd2,2,3216,0.7
3,gold,11,43754,0.2


## Done

Bronze verified, Silver rebuilt, dimensions versioned, Gold rebuilt, and gates green. Outputs are in `data/silver` and `data/gold`, ready for dbt, the dashboard, and the R reporting layer.